# Service Access Profiling

Read-only profiling of `pondolens.dev_bronze.service_access` (PH service access indicators 2021–2025).
Covers: schema, row counts, nulls/blanks, rescued data, indicators, regions, years, breakdowns, units, sources, value castability, duplicates, and cross-table compatibility.

In [0]:
-- 1. Sample preview
SELECT country, region, year, indicator, breakdown_type, breakdown_value, value, unit, source
FROM pondolens.dev_bronze.service_access
LIMIT 20

In [0]:
-- 2. Schema (all columns are STRING except _ingested_at)
DESCRIBE pondolens.dev_bronze.service_access

In [0]:
-- 3. Row count and source file
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT country, region, year, indicator, breakdown_type, breakdown_value) AS distinct_business_keys,
       COUNT(_rescued_data) AS rescued_rows,
       COUNT(DISTINCT _source_file) AS source_files
FROM pondolens.dev_bronze.service_access

In [0]:
-- 4. Null and blank counts per column
SELECT
  SUM(CASE WHEN country IS NULL OR country = '' THEN 1 ELSE 0 END)           AS country_nulls,
  SUM(CASE WHEN region IS NULL OR region = '' THEN 1 ELSE 0 END)             AS region_nulls,
  SUM(CASE WHEN year IS NULL OR year = '' THEN 1 ELSE 0 END)               AS year_nulls,
  SUM(CASE WHEN indicator IS NULL OR indicator = '' THEN 1 ELSE 0 END)       AS indicator_nulls,
  SUM(CASE WHEN breakdown_type IS NULL OR breakdown_type = '' THEN 1 ELSE 0 END) AS breakdown_type_nulls,
  SUM(CASE WHEN breakdown_value IS NULL OR breakdown_value = '' THEN 1 ELSE 0 END) AS breakdown_value_nulls,
  SUM(CASE WHEN value IS NULL OR value = '' THEN 1 ELSE 0 END)             AS value_blank,
  SUM(CASE WHEN unit IS NULL OR unit = '' THEN 1 ELSE 0 END)               AS unit_nulls,
  SUM(CASE WHEN source IS NULL OR source = '' THEN 1 ELSE 0 END)           AS source_nulls,
  SUM(CASE WHEN notes IS NULL OR notes = '' THEN 1 ELSE 0 END)             AS notes_blank
FROM pondolens.dev_bronze.service_access

In [0]:
-- 5. Rescued data check
SELECT _rescued_data, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY _rescued_data

In [0]:
-- 6. Distinct indicators with counts
SELECT indicator, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY indicator
ORDER BY cnt DESC

In [0]:
-- 7. Distinct regions with counts
SELECT region, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY region
ORDER BY cnt DESC

In [0]:
-- 8. Year coverage with blank-value counts
SELECT year,
       COUNT(*) AS total_rows,
       COUNT(value) AS non_null_values,
       SUM(CASE WHEN value = '' THEN 1 ELSE 0 END) AS blank_values,
       ROUND(SUM(CASE WHEN value = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_blank
FROM pondolens.dev_bronze.service_access
GROUP BY year
ORDER BY year

In [0]:
-- 9. Breakdown types and values
SELECT breakdown_type, breakdown_value, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY breakdown_type, breakdown_value
ORDER BY cnt DESC

In [0]:
-- 10. Unit distribution
SELECT unit, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY unit
ORDER BY cnt DESC

In [0]:
-- 11. Source distribution
SELECT source, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY source
ORDER BY cnt DESC

In [0]:
-- 12. Value castability (value is STRING — check if castable to double)
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN value != '' AND TRY_CAST(value AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS castable,
  SUM(CASE WHEN value != '' AND TRY_CAST(value AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS non_castable,
  SUM(CASE WHEN value = '' THEN 1 ELSE 0 END) AS blank,
  MIN(CASE WHEN TRY_CAST(value AS DOUBLE) IS NOT NULL THEN TRY_CAST(value AS DOUBLE) END) AS min_val,
  MAX(CASE WHEN TRY_CAST(value AS DOUBLE) IS NOT NULL THEN TRY_CAST(value AS DOUBLE) END) AS max_val,
  AVG(CASE WHEN TRY_CAST(value AS DOUBLE) IS NOT NULL THEN TRY_CAST(value AS DOUBLE) END) AS mean_val
FROM pondolens.dev_bronze.service_access

In [0]:
-- 13. Blank values by year × indicator (heatmap of data gaps)
SELECT year, indicator,
       COUNT(*) AS total,
       SUM(CASE WHEN value = '' THEN 1 ELSE 0 END) AS blank
FROM pondolens.dev_bronze.service_access
GROUP BY year, indicator
ORDER BY indicator, year

In [0]:
-- 14. Exact duplicates (full-row)
SELECT COUNT(*) AS duplicate_rows
FROM (
  SELECT *, COUNT(*) OVER (
    PARTITION BY country, region, year, indicator, breakdown_type, breakdown_value, value, unit, source, source_url, notes
  ) AS row_count
  FROM pondolens.dev_bronze.service_access
) t
WHERE row_count > 1

In [0]:
-- 15. Business key duplicates (country × region × year × indicator × breakdown_type × breakdown_value)
SELECT country, region, year, indicator, breakdown_type, breakdown_value, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY country, region, year, indicator, breakdown_type, breakdown_value
HAVING COUNT(*) > 1
ORDER BY cnt DESC

In [0]:
-- 16. Indicator overlap — check for similar/overlapping indicator names
SELECT indicator, breakdown_type, breakdown_value, unit, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
WHERE indicator LIKE '%sanitation%' OR indicator LIKE '%Internet%'
GROUP BY indicator, breakdown_type, breakdown_value, unit
ORDER BY indicator, breakdown_type

In [0]:
-- 17. Which indicators have Residence (rural/urban) breakdowns?
SELECT indicator, breakdown_type, COUNT(*) AS cnt
FROM pondolens.dev_bronze.service_access
GROUP BY indicator, breakdown_type
ORDER BY indicator, breakdown_type

In [0]:
-- 18. Region label consistency (check for naming mismatches vs other bronze tables)
SELECT DISTINCT region
FROM pondolens.dev_bronze.service_access
ORDER BY region

## Summary
Dataset: `pondolens.dev_bronze.service_access` — 400 rows, single source file (`PH_service_access_2021_2025.csv`), profiled read-only.

### What was checked
1–4. Sample preview, schema (22 columns — all STRING except `_ingested_at`), row count and source files, null/blank counts per column.
5. `_rescued_data` check.
6–9. Distinct indicators (9), regions (18), years (2021–2025), breakdown types/values (2 types, 7 values).
10–11. Unit distribution (7 units), source distribution (5 sources).
12–13. Value castability (STRING → DOUBLE), blank-value heatmap by year × indicator.
14–15. Exact and business-key duplicates.
16. Indicator overlap (similar/overlapping indicator names).
17. Residence (rural/urban) breakdown coverage by indicator.
18. Region label consistency.

### Key findings
- **Row count**: 400 rows across 5 years (80 rows/year). No exact duplicates, no business-key duplicates, zero `_rescued_data`. Clean row structure.
- **Indicators** (9): Coverage varies widely — `At least basic drinking-water access` and `Basic handwashing facilities` (105 each) down to `Internet use` (5). Two near-duplicate names to verify: `Basic sanitation access` (90) vs `At least basic sanitation access` (15) — are these different metrics or the same concept with different scope?
- **Regions** (18): `National` (105) + 17 regional. Label inconsistency: most use format "Region I (Ilocos Region)" but `MIMAROPA` and `NCR` lack the "Region IV-B" / "Region XIII" prefix. `CAR` also lacks full name.
- **Year coverage**:
  | Year | Rows | Blank values | % Blank |
  | --- | --- | --- | --- |
  | 2021 | 80 | 64 | 80% |
  | 2022 | 80 | 10 | 12.5% |
  | 2023 | 80 | 64 | 80% |
  | 2024 | 80 | 0 | 0% |
  | 2025 | 80 | 80 | 100% |
  - **2025 is entirely blank** — future year, no data published yet.
  - **2021 and 2023 are 80% blank** — odd years have significant gaps (likely survey/census cycle: PSA APIS data may only be available for even years).
  - **2024 is fully populated** — most reliable year.
- **Value**: Stored as STRING. All non-blank values cast to DOUBLE cleanly (0 non-castable). Range: 0 to ~100 (percentages). 218 of 400 rows (54.5%) are blank.
- **Breakdowns**: `Total` (350 rows, 7 values) and `Residence` (50 rows: Rural 25, Urban 25). Residence breakdown only exists for `Internet use, age 10+` and `Internet access at home`.
- **Units** (7): Each indicator has a specific unit (e.g., `% of families`, `% of population`, `% of households`). Unit is tied to indicator, not a free field.
- **Sources** (5): PSA APIS (270), PSA-DICT NICTHS (50), WHO/UNICEF JMP via World Bank WDI (45), Tracking SDG7 via World Bank WDI (30), ITU via World Bank WDI (5). Each indicator maps to one source.

### Conclusion — what to do in Silver

**Required Silver treatments**:
1. **Cast `value` to DOUBLE** — create `value_double` column; keep original `value` as `value_raw`. Filter blank strings to NULL.
2. **Handle blank/missing values** — 218/400 rows (54.5%) are blank. Decision needed: drop, keep as NULL, or flag with a `data_availability` column (e.g., `observed`, `missing`, `future_year`).
3. **Standardize region labels** — build a region crosswalk to match obligations/GAA/population/GRDP tables. `MIMAROPA` → `Region IV-B (MIMAROPA)`, `NCR` → `Region XIII (NCR)`, `CAR` → full name. Align with the region crosswalk from the obligations profiling notebook.
4. **Distinguish similar indicators** — clarify whether `Basic sanitation access` (90) and `At least basic sanitation access` (15) are different SDG metrics or duplicates. Add an `indicator_category` column (e.g., WASH, ICT, Energy) for grouping.
5. **Add `is_total_row` flag** — mark `region = 'National'` rows as national-level aggregates to prevent double-counting in regional analysis.
6. **Add `breakdown_category`** — classify `breakdown_type` as `aggregate` (Total) vs `disaggregated` (Residence) for easier filtering.
7. **Standardize `year`** — cast to INT for joins. Note that population table uses floats (`2020.0`); service_access uses strings.
8. **Source provenance** — preserve `source` and `source_url` but standardize source names for downstream lineage.

**Resolved questions**:

1. **Are 2021/2023 blanks due to the APIS survey cycle?** — Yes. PSA conducts APIS in years when FIES is *not* conducted; 2021 and 2023 were FIES years, so no APIS round exists for those reference years. This is more precise than calling APIS simply "biennial" — its schedule depends on FIES. Keep the values blank and note: "No APIS survey round for this reference year." We still need to check the actual file to confirm that every blank follows this pattern. *(Source: psa.gov.ph)*

2. **"Basic sanitation" vs "At least basic sanitation"** — These are distinct JMP ladder levels. "Basic" = improved, unshared facilities without safely managed classification. "At least basic" = basic plus safely managed sanitation. JMP groups both under "at least basic," but PSA's 2024 APIS did not separately classify safely managed sanitation (insufficient excreta treatment data) — its "basic sanitation" covers improved, unshared facilities only. Do not treat it as "basic only" or add a safely managed percentage to it. Keep the original PSA indicator label. Harmonize with another source only after checking definitions and denominators — PSA's percentage of families differs from JMP's percentage of population. *(Source: psa.gov.ph, JMP)*

3. **Will 2025 be backfilled?** — Do not assume a future 2025 APIS release. PSA conducted FIES for 2025 and APIS for 2026. A later release of 2026 APIS results would provide 2026 observations, not automatically fill 2025. The WASH release published in June 2025 reports 2024 APIS data. Retain 2025 as missing. Only backfill if an official source supplies a comparable 2025 reference-year value. *(Source: psa.gov.ph)*

4. **Which region crosswalk should we use?** — Use one shared crosswalk for obligations and service access. Reuse compatible mappings, but make stable codes the join key and descriptive labels the display field. Include: `source_region_label`, `region_code`, `standard_region_name`, `boundary_version` or applicable years. The obligations extraction uses two-digit analytical region prefixes, not full PSGC identifiers — preserve that distinction. A matching name or code alone does not guarantee matching boundaries; flag incompatible coverage rather than forcing the join.